# **CAT VS DOG CLASSIFIER**

In [82]:
import os
import torch
from torch.utils.data import Dataset,DataLoader
from PIL import Image
import torch.optim as optim
import torch.nn as nn
from torchvision import transforms

# **IMPORT DATA FROM KAGGLE**

In [ ]:
#enter kaggle credentials
os.environ['KAGGLE_USERNAME'] = 'name1'
os.environ['KAGGLE_KEY'] = 'k_key'

In [ ]:
#load the data and extract
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset
!unzip dog-and-cat-classification-dataset.zip

In [ ]:
#check directories
!ls

In [ ]:
!ls PetImages

# **CREATE A CUSTOM DATASET CLASS**

In [83]:
#create a class which helps you customize your data
class CatDogDataset(Dataset):
  #here we convert files and dir into image paths and labels
  def __init__(self,root_dir,transform=None):
    self.root_dir=root_dir
    self.transform=transform
    self.img_paths=[]
    self.labels=[]
    self.classes={"Cat":0,"Dog":1}
    for path in os.listdir(root_dir):
      class_folder=os.path.join(root_dir,path)
      if not os.path.isdir(class_folder):
        continue
      if not os.listdir(class_folder):
        continue
      for img_name in os.listdir(class_folder):
        img_path=os.path.join(class_folder,img_name)
        self.img_paths.append(img_path)
        self.labels.append(self.classes[path])
#gives length of dataset
  def __len__(self):
    return len(self.img_paths)
#Indexing
  def __getitem__(self,idx):
    img_path=self.img_paths[idx]
    label=self.labels[idx]
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Error loading image {img_path}: {e}. Skipping to next image.")
        return self.__getitem__((idx + 1) % len(self.img_paths))
    if self.transform:
      image=self.transform(image)
    return image,label

# **ADD TRANSFORMATIONS**

In [157]:
#Adding various transformation to data for training
transform = transforms.Compose([
    transforms.Resize((64,64)),

    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),

    transforms.ColorJitter(
        brightness=0.8,
        contrast=0.8,
        saturation=0.8,
        hue=0.2
    ),

    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3)
    ], p=0.3),

    transforms.RandomGrayscale(p=0.3),

    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# **CREATE MODEL**

In [131]:
#creating a CNN model
from torch.nn.modules.pooling import MaxPool2d
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv=nn.Sequential(
        nn.Conv2d(3,16,3),
        nn.ReLU(),
        MaxPool2d(2),
        nn.Conv2d(16,32,3),
        nn.ReLU(),
        MaxPool2d(2),
        nn.Conv2d(32,64,3),
        nn.ReLU(),
        MaxPool2d(2),
        nn.Conv2d(64,128,3),
        nn.ReLU(),
        MaxPool2d(2)
    )
    self.fc = None

  def forward(self, x):
    x = self.conv(x)
    #print("After conv:", x.shape)
    x = torch.flatten(x, 1)
    if self.fc is None:
      self.fc = nn.Linear(x.shape[1], 2).to(x.device)
    x = self.fc(x)
    return x


# **INITIALISATION**

In [132]:
#Initialise model
model = CNN()


tensor([[-0.0656, -0.0434]], grad_fn=<AddmmBackward0>)

In [158]:
#load dataset
dataset=CatDogDataset("PetImages",transform=transform)

In [159]:
#split data for training and validation 80-20
dataset = CatDogDataset("PetImages", transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [160]:
#Initialise dataloader object
loader=DataLoader(dataset,batch_size=32,shuffle=True,num_workers=0)

In [161]:
#Initialise optimizer
optimizer=optim.Adam(model.parameters(),lr=0.001)
loss_fn=nn.CrossEntropyLoss()


In [162]:
#Change device to gpu if available for speed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

# **TRAINING LOOP**

In [171]:
#Train the model
epochs=12
for epoch in range(epochs):
  total_loss=0
  for img,labels in loader:
    output=model(img.to(device))
    loss=loss_fn(output,labels.to(device))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  avg_loss = total_loss / len(loader)
  print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 0.3813
Epoch 2, Loss: 0.3756
Epoch 3, Loss: 0.3710
Epoch 4, Loss: 0.3751
Epoch 5, Loss: 0.3705
Epoch 6, Loss: 0.3696
Epoch 7, Loss: 0.3631
Epoch 8, Loss: 0.3616
Epoch 9, Loss: 0.3588
Epoch 10, Loss: 0.3524
Epoch 11, Loss: 0.3514
Epoch 12, Loss: 0.3554


# **EVALUATION**

In [172]:
#Find validatrion loss
model.eval()
val_loss = 0

with torch.no_grad():
    for img, labels in val_loader:
        img = img.to(device)
        labels = labels.to(device)

        output = model(img)
        loss = loss_fn(output, labels)
        val_loss += loss.item()

val_loss /= len(val_loader)
print("Validation Loss:", val_loss)

Validation Loss: 0.34315626826255946


## **TESTING**

In [173]:
#Create test transform to match training
test_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [179]:
from PIL import Image
#load test file
img_path = "/content/cat2.png"
image = Image.open(img_path).convert("RGB")
image = test_transform(image)
image = image.unsqueeze(0)
image = image.to(device)

In [180]:
#find prediction
model.eval()

with torch.no_grad():
    output = model(image)
    _, pred = torch.max(output, 1)

In [181]:
classes = ["Cat", "Dog"]

print("Prediction:", classes[pred.item()])

Prediction: Cat


In [182]:
import torch.nn.functional as F
#Check model confidence
probs = F.softmax(output, dim=1)
print("Confidence:", probs)

Confidence: tensor([[0.6414, 0.3586]], device='cuda:0')
